In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 50. Week 34 — Sequence models: LSTM and causal TCN

## 学習目標

- recurrent stateとcausal convolutionの情報経路を比較できる
- LSTMのfour gatesとcell updateを書ける
- TCNのreceptive fieldをkernel/dilationから計算できる
- 同じtoken sequence・width・linear probeでrepresentationを比較できる

## 前提知識

- Week 33のcomputational graph
- convolutionとrecurrenceの基本
- previous-filing-only text contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 50


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask

assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
assert set(fixture.partitions) == {"inner_train", "inner_validation"}

print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("numeric / sequence shape:", fixture.numeric_features.shape, fixture.token_hashes.shape)
print("locked outer rows present: False")
print("fixture hash lineage:", fixture.provenance)

fixture rows: 256
inner train / validation: 192 64
numeric / sequence shape: (256, 12) (256, 128)
locked outer rows present: False
fixture hash lineage: {'panel_artifact_sha256': '6c6008c2f28c30299e15e37613cfb0b3b22e8fd283858f5b459227c7e4a412a8', 'previous_filing_sidecar_sha256': '9ff2efef335357ff53bb1e4ba5c57f4b2e8799fc4ee5d830c55843a50026fbbc', 'normalized_manifest_sha256': '1283b9cb0992cfd2caaa942f6c869e212762c90a9abbc9a050173f5e3963daba', 'preanalysis_contract_sha256': 'fbe69fdf3b3bccba7fab70bcbb726d0df61685901cc0322d76fc66be1d7bbd6e'}


## 1. LSTMとTCN

LSTM cellは

$$
f_t=\sigma(W_fx_t+U_fh_{t-1}+b_f),\quad
i_t=\sigma(W_ix_t+U_ih_{t-1}+b_i),
$$

$$
c_t=f_t\odot c_{t-1}+i_t\odot\tanh(W_cx_t+U_ch_{t-1}+b_c),
\quad h_t=o_t\odot\tanh c_t.
$$

TCNはcausal paddingにより時点 (t) の出力が (x_{1:t}) だけに依存する。kernel幅 (k)、layer数 (L)、dilation (d_l) ならreceptive fieldは (1+(k-1)\sum_l d_l) である。

In [4]:
bptt_rng = task_rng(1)
bptt_embeddings = bptt_rng.normal(size=(3, 4, 2))
bptt_target = bptt_rng.normal(size=3)
bptt_parameters = qt.initialize_lstm(2, 2, rng=bptt_rng)
bptt_audit = qt.check_lstm_gradients(
    bptt_parameters,
    bptt_embeddings,
    bptt_target,
    step=1e-6,
    tolerance=5e-5,
)
assert bptt_audit.passed
print("BPTT maximum relative gradient error:", bptt_audit.maximum_relative_error)

BPTT maximum relative gradient error: 3.8587774365107155e-08


In [5]:
embedding_width = 8
representation_width = 12
embeddings = qt.token_embedding(fixture.token_hashes, embedding_width, seed=20260811)
architecture_rng = task_rng(2)

lstm_input = architecture_rng.normal(
    scale=1.0 / np.sqrt(embedding_width), size=(embedding_width, 4 * representation_width)
)
lstm_recurrent = architecture_rng.normal(
    scale=1.0 / np.sqrt(representation_width),
    size=(representation_width, 4 * representation_width),
)
lstm_bias = np.zeros(4 * representation_width)
lstm_representation = qt.lstm_encode(embeddings, lstm_input, lstm_recurrent, lstm_bias)

tcn_kernels = architecture_rng.normal(
    scale=1.0 / np.sqrt(3 * embedding_width),
    size=(3, embedding_width, representation_width),
)
tcn_representation = qt.temporal_convolution_encode(
    embeddings, tcn_kernels, np.zeros(representation_width)
)
mean_representation = embeddings.mean(axis=1)

assert lstm_representation.shape == (fixture.targets.size, representation_width)
assert tcn_representation.shape == (fixture.targets.size, representation_width)
print("effective sequence length:", fixture.token_hashes.shape[1])
print("single-layer TCN receptive field:", 3)

effective sequence length: 128
single-layer TCN receptive field: 3


## 2. Frozen representation probe

ここではBPTT candidateを学習したと主張しない。forward recurrence/convolutionを透明に確認した後、同じridge probeを各固定representationへ当て、data scaleに対してarchitecture差を判断する準備をする。正式candidateはpre-registered run/epoch/parameter budgetでend-to-end学習する。

In [6]:
representations = {
    "mean_embedding": mean_representation,
    "random_lstm": lstm_representation,
    "random_tcn": tcn_representation,
}
probe_rows = []
probe_predictions = {}
for name, representation in representations.items():
    probe = qt.fit_sparse_ridge(
        representation[train_mask], fixture.targets[train_mask], ridge=1.0
    )
    prediction = probe.predict(representation[validation_mask])
    probe_predictions[name] = prediction
    probe_rows.append(
        {
            "representation": name,
            "width": representation.shape[1],
            **qt.regression_error_table(
                fixture.targets[validation_mask],
                prediction,
                np.asarray(fixture.entity_ids)[validation_mask],
            ),
        }
    )
probe_table = pd.DataFrame(probe_rows)
display(probe_table)

fig = go.Figure()
for name, prediction in probe_predictions.items():
    fig.add_scatter(
        x=fixture.targets[validation_mask],
        y=prediction,
        mode="markers",
        name=name,
    )
fig.update_layout(
    title="Same-data frozen representation probes",
    xaxis_title="Observed target",
    yaxis_title="Probe prediction",
    template="plotly_white",
)
fig.show()

,representation,width,mae,median_absolute_error,rmse,company_macro_mae
0,mean_embedding,8,0.053111,0.024188,0.118754,0.046991
1,random_lstm,12,0.053209,0.021329,0.120407,0.046769
2,random_tcn,12,0.052386,0.024885,0.118717,0.046313


## 3. 失敗モード

- bidirectional contextやtarget filingをprevious-filing featureへ混ぜる
- causal paddingなしのconvolutionをforecast modelと呼ぶ
- LSTM/TCNでtokenization、sequence length、probeを変える
- frozen random featureの結果をtrained architectureの結果と呼ぶ
- BPTTのtruncation lengthを隠す

## 4. 段階別演習

### 基礎

1. forget gateが1、input gateが0のcell updateを説明せよ。
2. kernel 5、dilation 1/2/4のreceptive fieldを求めよ。

### 標準

3. sequence後半だけを変え、causal TCNの過去出力が不変なtestを書け。
4. 同じparameter budgetでLSTM widthとTCN channel widthを決めよ。

### 研究

5. chunk平均がdocument順序情報をどこで失うか監査せよ。

## 5. Exit Criteria

- [ ] LSTM four gatesを実装した
- [ ] TCNのcausal contractをtestした
- [ ] sequence、width、probe budgetを揃えた
- [ ] frozen probeとend-to-end candidateを区別した
- [ ] effective contextとdocument lengthを報告した

## 6. 出典


- [Hochreiter and Schmidhuber (1997), Long Short-Term Memory](https://www.bioinf.jku.at/publications/older/2604.pdf)
- [Bai, Kolter, and Koltun (2018), An Empirical Evaluation of Generic Convolutional and Recurrent Networks](https://arxiv.org/abs/1803.01271)
- [Vaswani et al. (2017), Attention Is All You Need](https://arxiv.org/abs/1706.03762)